# Polynomial Regression — Intuition

**Goal.** Build a mental picture of polynomial regression *before* touching any math. Three questions, answered with pictures only:

1. Why isn't a straight line enough sometimes?
2. What is a "polynomial feature" and how does it fix the problem?
3. What happens when we go too far — degree 2, 5, 9, 15?

No formulas here. The math lives in `02_mathematics.ipynb`.

**One-line preview.** Polynomial regression = linear regression on a *transformed* feature: replace x by (1, x, x^2, …, x^d). Everything else (OLS, normal equations, gradient descent) carries over unchanged.

**Prerequisites.** `01_linear_regression/01_intuition.ipynb` — bowl-shaped loss, residuals, the case where a straight line fails.

**Stage map.** `01_intuition` → `02_mathematics` → `03_optimization` → `04_statistics` → `05_hands_on_programming`.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random

import numpy as np
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. The data — a curve, not a line

Same setup as last folder but with a *curved* signal: y is generated as a sine wave plus noise. A straight line cannot follow this no matter where you place it.

In [ ]:
def true_fn(x):
    return np.sin(1.6 * x) + 0.4 * x

n = 30
x = rng.uniform(-3, 3, size=n)
y = true_fn(x) + rng.normal(0, 0.25, size=n)

xs = np.linspace(-3.2, 3.2, 300)

# Linear fit attempt.
slope, intercept = np.polyfit(x, y, 1)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(xs, true_fn(xs), color="black", ls="--", label="true f(x)")
ax.scatter(x, y, alpha=0.7, edgecolor="k", label="observed (x, y)")
ax.plot(xs, slope * xs + intercept, color="crimson", lw=2, label="best straight line")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("A straight line cannot follow a curve")
ax.legend(fontsize=8)
plt.show()

## 2. The polynomial trick

We do *not* invent a new algorithm. We just give linear regression more features to work with.

Original feature: x (one number per observation).

Polynomial features (degree d): the *vector* (1, x, x^2, x^3, …, x^d). The model now reads

    $\hat{y}$ = $\theta_0$ + $\theta_1$ x + $\theta_2$ x^2 + … + $\theta_d$ x^d.

This is *non-linear in x* but **linear in the parameters $\theta^*$* — so OLS still applies, with a wider design matrix. Below: degree 1 (a line), degree 3 (curves smoothly), degree 9 (already starts wiggling).

In [ ]:
def fit_poly(x, y, d):
    """Fit polynomial of degree d via np.polyfit (which is OLS on Vandermonde columns)."""
    coefs = np.polyfit(x, y, d)
    return np.poly1d(coefs)

degrees_demo = [1, 3, 9]
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), sharey=True)
for ax, d in zip(axes, degrees_demo):
    poly = fit_poly(x, y, d)
    ax.plot(xs, true_fn(xs), color="black", ls="--", lw=1, label="true f")
    ax.scatter(x, y, alpha=0.7, edgecolor="k")
    ax.plot(xs, poly(xs), color="crimson", lw=2, label=f"degree {d}")
    ax.set_xlabel("x")
    ax.set_title(f"degree {d}")
    ax.legend(fontsize=8)
axes[0].set_ylabel("y")
plt.suptitle("More features → more flexibility (and eventually too much)")
plt.tight_layout()
plt.show()

**Reading.** Degree 1 is the straight line from §1, still bad. Degree 3 captures the curvature. Degree 9 is already starting to chase noise at the edges — a hint of what comes next.

## 3. The bias–variance trade-off

More features always reduce **training** error — the model can always fit the training points more closely. But what matters is how well it fits **new** points. Two failure modes:

- **Underfit** (degree too low) — the model misses real structure. Both training and test errors are large. *High bias.*
- **Overfit** (degree too high) — the model memorises noise. Training error is tiny, test error is huge. *High variance.*

We simulate this by splitting the data into train / test, fitting each degree on the train set, and plotting both errors as a function of degree.

In [ ]:
# Generate a larger sample, split into train / test so we have something to evaluate on.
N = 60
x_all = rng.uniform(-3, 3, size=N)
y_all = true_fn(x_all) + rng.normal(0, 0.25, size=N)

perm = rng.permutation(N)
n_train = N // 2
tr_idx, te_idx = perm[:n_train], perm[n_train:]
x_tr, y_tr = x_all[tr_idx], y_all[tr_idx]
x_te, y_te = x_all[te_idx], y_all[te_idx]

degrees = np.arange(1, 16)
mse_tr, mse_te = [], []
for d in degrees:
    poly = fit_poly(x_tr, y_tr, d)
    mse_tr.append(np.mean((y_tr - poly(x_tr)) ** 2))
    mse_te.append(np.mean((y_te - poly(x_te)) ** 2))
mse_tr, mse_te = np.array(mse_tr), np.array(mse_te)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(degrees, mse_tr, "o-", color="steelblue", label="train MSE")
ax.plot(degrees, mse_te, "o-", color="crimson",  label="test MSE")
ax.set_yscale("log")
ax.set_xlabel("polynomial degree d")
ax.set_ylabel("MSE (log scale)")
ax.set_title("Bias–variance trade-off: U-shaped test error")
ax.axvline(degrees[np.argmin(mse_te)], color="black", ls=":", lw=1,
           label=f"best test deg = {degrees[np.argmin(mse_te)]}")
ax.legend()
plt.show()

**Reading.**

- Training MSE drops monotonically — more flexibility always helps the training fit.
- Test MSE is **U-shaped**. Left side: underfit (high bias, the model is too rigid). Right side: overfit (high variance, the model has chased noise).
- The sweet spot is the bottom of the U — usually a low degree. "More features" is not a free lunch.

Pictures of overfit vs. underfit, side by side:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), sharey=True)
for ax, (d, label) in zip(axes, [(1, "underfit"), (3, "about right"), (15, "overfit")]):
    poly = fit_poly(x_tr, y_tr, d)
    ax.plot(xs, true_fn(xs), color="black", ls="--", lw=1, label="true f")
    ax.scatter(x_tr, y_tr, alpha=0.6, edgecolor="k", label="train")
    ax.scatter(x_te, y_te, alpha=0.6, marker="x", color="orange", label="test")
    ax.plot(xs, poly(xs), color="crimson", lw=2)
    ax.set_xlabel("x")
    ax.set_title(f"{label}  (degree {d})")
    ax.set_ylim(-3, 3)
    ax.legend(fontsize=7, loc="upper left")
axes[0].set_ylabel("y")
plt.tight_layout()
plt.show()

## 4. Why "linear" regression handles all of this

The crucial observation: the model

    $\hat{y}$ = $\theta_0$ + $\theta_1$ x + $\theta_2$ x^2 + … + $\theta_d$ x^d

is **non-linear in x**, but **linear in $\theta^*$*. The OLS machinery — closed form, gradient descent, convexity, Gauss–Markov — only cares about linearity *in the parameters*. So replacing each $x_i$ with the row (1, $x_i$, $x_i$^2, …, $x_i$^d) gives a new, wider design matrix $\tilde{X}$, and we are back to ordinary least squares on $\tilde{X}$.

Everything we proved in `01_linear_regression/02_mathematics.ipynb` applies, but on $\tilde{X}$ instead of X. That is what makes the next four notebooks short.

## Takeaway

- A straight line cannot follow a curve. Polynomial features (1, x, x^2, …, x^d) give a *non-linear-in-x* model that is still *linear in the parameters*, so OLS still applies — just on a wider design matrix.
- Increasing the degree d always reduces **training** error. But **test** error is U-shaped: too small d under-fits (bias), too large d over-fits (variance). The bottom of the U is where the model generalises best.
- Polynomial regression is therefore not a new algorithm — it is OLS on a *transformed* feature space. The work shifts from "how do I fit?" to "how do I choose d?".

Next: `02_mathematics.ipynb` formalises the polynomial *feature map* $\Phi$(x), restates OLS on $\Phi$(X), and explains why high-degree designs are numerically unfriendly (the Vandermonde condition number).